# Section 00.03D — Hugging Face Overview

**Data Science and Machine Learning Course**

---

## Learning Objectives

By the end of this section you will be able to:
- Understand what Hugging Face is and how it fits into a data science workflow
- Use the `transformers` library to run pretrained models with a single line of code
- Load models and tokenizers from the Hugging Face Hub
- Use the `datasets` library to access thousands of ready-to-use datasets
- Run inference using Hugging Face Pipelines
- Understand when to fine-tune versus use a pretrained model as-is
- Use Hugging Face Spaces to deploy interactive demos
- Access models via the Inference API without local GPU
- Compare Hugging Face with Kaggle and Colab for different use cases

---

## 1. What is Hugging Face?

**Hugging Face** is an AI company and open-source platform that has become the central hub for modern machine learning, particularly in NLP, computer vision, and multimodal AI.

It provides three things that work together:

| Component | What it is | Analogy |
|-----------|-----------|---------|
| **Hub** | Repository of 500 000+ pretrained models and datasets | GitHub for ML models |
| **Libraries** | `transformers`, `datasets`, `peft`, `diffusers`, etc. | PyPI for ML workflows |
| **Spaces** | Cloud-hosted interactive demos (Gradio / Streamlit) | Heroku for ML apps |

**How it differs from Kaggle and Colab:**

| Feature | Google Colab | Kaggle | Hugging Face |
|---------|-------------|--------|--------------|
| Primary purpose | Run notebooks | Competitions + data | Models + deployment |
| Notebook environment | Yes | Yes | Yes (via Spaces) |
| Dataset access | Manual / Drive | One-click attach | `datasets` library |
| Model sharing | No | No | Central to the platform |
| Demo deployment | No | No | Spaces (Gradio/Streamlit) |
| Free GPU | Yes (limited) | 30 hrs/week | Via Colab or local |

> **Key idea:** Hugging Face is not a notebook environment — it is a **model and dataset ecosystem**. You use it *with* Jupyter, Colab, or Kaggle to access state-of-the-art pretrained models.

---

## 2. Installing the Core Libraries

Hugging Face libraries are available on PyPI and work in any Python environment — local, Colab, or Kaggle.

In [ ]:
# Install core Hugging Face libraries
# Run this in Colab, Kaggle, or a local terminal

# !pip install transformers datasets huggingface_hub

# Optional — for audio, image, and training workflows:
# !pip install accelerate evaluate timm torchaudio

print("transformers  — pretrained models and tokenizers")
print("datasets      — ready-to-use NLP/CV datasets")
print("huggingface_hub — interact with the Hub (login, upload, download)")

---

## 3. The Hub — Finding Models and Datasets

The **Hugging Face Hub** at [huggingface.co](https://huggingface.co) is a searchable repository of models, datasets, and Spaces.

**Finding a model:**
1. Go to **huggingface.co/models**
2. Filter by task (e.g., *text-classification*, *image-segmentation*, *text-generation*)
3. Filter by library (PyTorch, TensorFlow, JAX) or language
4. Click a model card to read documentation, see example code, and try it in the browser

**Key fields on a model card:**
- **Model ID** — `username/model-name` — the identifier used in code
- **Task** — what the model is designed for
- **Spaces** — live demos built on top of the model
- **Files and versions** — the actual weights stored on the Hub

In [ ]:
from huggingface_hub import list_models

# Search the Hub programmatically — filter by task
models = list(list_models(task='text-classification', limit=5))
for m in models:
    print(m.id)

---

## 4. Pipelines — Instant Inference in One Line

The `pipeline` function is the fastest way to run a pretrained model. It handles tokenization, inference, and decoding automatically.

In [ ]:
from transformers import pipeline

# Sentiment analysis — downloads a small default model on first run
classifier = pipeline('sentiment-analysis')
result = classifier("Hugging Face makes machine learning much more accessible.")
print(result)

The output is a list of dicts: `[{'label': 'POSITIVE', 'score': 0.9998}]`

`pipeline` accepted a task name and handled everything: downloading weights, tokenizing input, running the model, and decoding the output.

In [ ]:
# Common pipeline tasks — reference
#
# pipeline('sentiment-analysis')         → text classification
# pipeline('text-generation')            → generate text (GPT-style)
# pipeline('summarization')              → summarise long text
# pipeline('translation_en_to_fr')       → machine translation
# pipeline('question-answering')         → extract answer from context
# pipeline('fill-mask')                  → fill in [MASK] tokens (BERT-style)
# pipeline('ner')                        → named entity recognition
# pipeline('zero-shot-classification')   → classify without training
# pipeline('image-classification')       → classify images
# pipeline('automatic-speech-recognition') → transcribe audio

# Specify a model explicitly to override the default
# pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

print("Use the task name to get a default model, or pass model= for a specific one.")

---

## 5. Loading Models and Tokenizers Directly

For more control than `pipeline` allows, load the tokenizer and model separately.
This is the standard pattern when you need to inspect outputs, batch inputs, or integrate into a larger system.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = 'distilbert-base-uncased-finetuned-sst-2-english'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
text = "This course is surprisingly well structured."

# Tokenize — converts text to token IDs the model understands
inputs = tokenizer(text, return_tensors='pt')
print("Token IDs:", inputs['input_ids'])
print("Tokens:   ", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

In [ ]:
import torch

# Run the model and decode the prediction
with torch.no_grad():
    outputs = model(**inputs)

predicted_class = outputs.logits.argmax(dim=1).item()
label = model.config.id2label[predicted_class]
print("Predicted label:", label)

**Pattern summary:**

| Step | Code |
|------|------|
| Load tokenizer | `AutoTokenizer.from_pretrained(model_name)` |
| Load model | `AutoModel*.from_pretrained(model_name)` |
| Tokenize input | `tokenizer(text, return_tensors='pt')` |
| Run inference | `model(**inputs)` inside `torch.no_grad()` |
| Decode output | `outputs.logits.argmax()` + `id2label` |

`Auto*` classes automatically select the right architecture — you don't need to know whether it's BERT, RoBERTa, or DistilBERT.

---

## 6. The `datasets` Library

The `datasets` library gives programmatic access to thousands of curated datasets — NLP, computer vision, audio, and tabular — without manual downloading.

In [ ]:
from datasets import load_dataset

# Load the IMDb sentiment dataset — downloads and caches automatically
dataset = load_dataset('imdb', split='train')
print(type(dataset))
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)

In [ ]:
# Inspect the first example
print(dataset[0]['text'][:200])
print("Label:", dataset[0]['label'])  # 0 = negative, 1 = positive

In [ ]:
import pandas as pd

# Convert to a Pandas DataFrame for familiar exploration
df = dataset.to_pandas()
print(df.head())
print("\nLabel distribution:")
print(df['label'].value_counts())

**Key `datasets` operations:**

```python
# Load with train/test splits
dataset = load_dataset('imdb')          # returns DatasetDict with 'train' and 'test'

# Filter rows
positives = dataset.filter(lambda x: x['label'] == 1)

# Apply a function to all rows
dataset = dataset.map(lambda x: tokenizer(x['text'], truncation=True))

# Shuffle and split
dataset = dataset.shuffle(seed=42).select(range(1000))
```

---

## 7. The Inference API — No Local GPU Required

For quick experiments, Hugging Face provides a hosted **Inference API** — you send text to a model on their servers and get predictions back.

No GPU needed locally. Rate-limited on the free tier; faster with a paid token.

In [ ]:
from huggingface_hub import InferenceClient

# Requires a free HF account — get a token at huggingface.co/settings/tokens
# client = InferenceClient(token='hf_your_token_here')

# Text classification via the API — model runs on HF servers
# result = client.text_classification(
#     "The bootcamp examples are really helpful.",
#     model='distilbert-base-uncased-finetuned-sst-2-english'
# )
# print(result)

# Text generation via the API
# result = client.text_generation(
#     "Data science is important because",
#     model='gpt2',
#     max_new_tokens=50
# )
# print(result)

print("Inference API: model runs on HF servers — no local GPU needed.")
print("Get your token at: huggingface.co/settings/tokens")

---

## 8. Fine-Tuning a Pretrained Model

**Fine-tuning** adapts a pretrained model to a specific task or domain by continuing training on your own labelled data.

When to fine-tune vs use as-is:

| Situation | Approach |
|-----------|---------|
| Task matches a standard pipeline | Use `pipeline` directly |
| Need a specific domain (medical, legal, code) | Fine-tune on domain data |
| Custom label set not in base model | Fine-tune with your labels |
| Need highest possible accuracy | Fine-tune on task-specific data |

In [ ]:
# Fine-tuning skeleton using Hugging Face Trainer (reference template)
# Requires GPU — run in Colab or Kaggle with GPU enabled

# from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
# from datasets import load_dataset

# dataset   = load_dataset('imdb')
# tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
# model     = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# def tokenize(batch):
#     return tokenizer(batch['text'], truncation=True, padding=True)

# tokenized = dataset.map(tokenize, batched=True)

# args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=2,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     evaluation_strategy='epoch',
#     save_strategy='epoch',
#     load_best_model_at_end=True,
# )

# trainer = Trainer(model=model, args=args,
#                   train_dataset=tokenized['train'],
#                   eval_dataset=tokenized['test'])
# trainer.train()

print("Fine-tuning requires a GPU — uncomment and run in Colab or Kaggle with GPU enabled.")

---

## 9. Hugging Face Spaces — Deploying a Demo

**Spaces** lets you deploy interactive ML demos for free using Gradio or Streamlit — no server management required.

**To create a Space:**
1. Go to **huggingface.co/spaces** → **Create new Space**
2. Choose **Gradio** or **Streamlit** as the SDK
3. Write your app in `app.py` — push to the Space's Git repository
4. The Space builds and becomes publicly accessible at `huggingface.co/spaces/username/space-name`

In [ ]:
# Minimal Gradio app for a sentiment classifier (save as app.py in your Space)

# import gradio as gr
# from transformers import pipeline
#
# classifier = pipeline('sentiment-analysis')
#
# def predict(text):
#     result = classifier(text)[0]
#     return f"{result['label']} ({result['score']:.2%})"
#
# demo = gr.Interface(
#     fn=predict,
#     inputs=gr.Textbox(label="Enter text"),
#     outputs=gr.Textbox(label="Sentiment"),
#     title="Sentiment Analyser",
#     description="Powered by DistilBERT fine-tuned on SST-2"
# )
#
# demo.launch()

print("Save the above as app.py in a Hugging Face Space to get a live public demo.")

---

## 10. Logging In and Pushing to the Hub

You can push your own models and datasets to the Hub to share with the community or keep private.

In [ ]:
from huggingface_hub import login

# Log in with your HF token (get it at huggingface.co/settings/tokens)
# login(token='hf_your_token_here')

# Or log in interactively — prompts for token input
# login()

# After training, push a fine-tuned model to your Hub profile
# model.push_to_hub('your-username/my-sentiment-model')
# tokenizer.push_to_hub('your-username/my-sentiment-model')

# Push a dataset
# from datasets import Dataset
# my_dataset = Dataset.from_pandas(df)
# my_dataset.push_to_hub('your-username/my-dataset')

print("push_to_hub() uploads your model/dataset to huggingface.co for sharing or reuse.")

---

## 11. Mini Exercise — Run Your First Pipeline

Work through the following to confirm the environment is set up correctly.

In [ ]:
# Step 1: Confirm transformers is installed and check the version
import transformers
print("transformers version:", transformers.__version__)

In [ ]:
from transformers import pipeline

# Step 2: Run a zero-shot classifier — no task-specific training needed
classifier = pipeline('zero-shot-classification')

text = "The quarterly revenue exceeded expectations by 12 percent."
labels = ['finance', 'sports', 'politics', 'technology']

result = classifier(text, candidate_labels=labels)
print("Text:", text)
print("Top label:", result['labels'][0], f"({result['scores'][0]:.2%})")

In [ ]:
# Step 3: Use Alt+Enter to add a cell below and try a different task
# Ideas:
#   pipeline('summarization')("Long article text here...", max_length=60)
#   pipeline('ner')("Marie Curie was born in Warsaw in 1867.")

---

## Key Takeaways

- Hugging Face is a **model and dataset ecosystem**, not a notebook environment — use it with Jupyter, Colab, or Kaggle.
- The **Hub** hosts 500 000+ pretrained models searchable by task, language, and architecture.
- `pipeline('task-name')` is the fastest way to run a pretrained model — one line handles tokenization, inference, and decoding.
- `Auto*` classes (`AutoTokenizer`, `AutoModelForSequenceClassification`) let you load any model without knowing its architecture.
- The `datasets` library provides one-line access to thousands of curated datasets; convert to Pandas with `.to_pandas()`.
- The **Inference API** runs models on HF servers — no local GPU needed for experimentation.
- **Fine-tune** when standard pipelines do not match your domain or label set; use `Trainer` with a GPU runtime.
- **Spaces** (Gradio / Streamlit) lets you deploy interactive demos publicly for free with minimal code.
- `push_to_hub()` shares fine-tuned models and datasets with the community or keeps them private.

---

*Review: **03 — Jupyter**, **03A — Google Colab**, **03B — Kaggle** for the full environment overview.*